# Q4 — Does sampling training subsets improve robustness?

Hypothesis: subset averaging helps with small, noisy, or heterogeneous data, but can waste information when the full training set is already sufficient. The controlled comparison fixes the family and parameter prior while changing only subset support.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while root != root.parent and not (root / 'bayesian_predictive_model_averaging').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from sklearn.metrics import log_loss
from EXPERIMENTS.common import classification_data, split_data, fit_classifier

In [2]:
X, y = classification_data(seed=17, kind='nonlinear')
X_train, X_test, y_train, y_test = split_data(X, y, seed=17)
settings = {
    'default subset prior': {},
    'full training subset': {'min_subset_size': len(y_train), 'max_subset_size': len(y_train)},
}
results = {}
for label, options in settings.items():
    model = fit_classifier(X_train, y_train, seed=17, **options)
    results[label] = {
        'log_loss': log_loss(y_test, model.predict_proba(X_test)),
        'ess_fraction': model.effective_sample_size_fraction_,
        'mean_subset_size': sum(draw['subset_size'] for draw in model.get_model_draws()) / model.n_estimators_,
    }
results

{'default subset prior': {'log_loss': 0.44280161168948806,
  'ess_fraction': 0.998548848596264,
  'mean_subset_size': 192.875},
 'full training subset': {'log_loss': 0.4166393058702387,
  'ess_fraction': 0.9979294547625442,
  'mean_subset_size': 450.0}}

Repeat after injecting outliers and label noise. Add fixed narrow and broad subset-size priors, plus bootstrap or subsampling baselines with the same number of fits. Report both test metrics and prediction variability across data resamples.

## Conclusion from the executed starter run

**Status: not falsified; the predicted trade-off is visible.** On this baseline dataset, the default subset prior produced log loss 0.4428, worse than 0.4166 when every draw used the full training set. This is compatible with the hypothesis that subsets can hurt when data are plentiful, but it provides no evidence yet that subset averaging helps under noise, outliers, or small samples.